# Summarize One Model BT Results

This notebook loads one `experiment.json` from `results/` and outputs a table with:
- One overall row for the model
- One row per `task_type`

Metrics:
- **Task accuracy** = completed tasks / total tasks (and %)
- **Structure adherence** = valid trees / total trees (and %)

In [6]:
from pathlib import Path
import json
import pandas as pd

In [7]:
# Resolve repo root robustly (works even if notebook cwd is elsewhere)
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src" / "tasks" / "tasks_100.json").exists() and (p / "results").exists():
            return p
    raise FileNotFoundError(
        "Could not find repo root. Start Jupyter from the project directory or set repo_root manually."
    )

repo_root = find_repo_root(Path.cwd())

# Set this to your experiment.json path relative to repo root (or absolute path)
experiment_rel = Path("results/wall_bt_2026-04-29_14-41-35/Qwen/Qwen2.5-3B-Instruct/experiment.json")
experiment_path = experiment_rel if experiment_rel.is_absolute() else (repo_root / experiment_rel)

# Task catalog used for this experiment
tasks_path = repo_root / "src" / "tasks" / "tasks_100.json"

experiment = json.loads(experiment_path.read_text(encoding="utf-8"))
tasks = json.loads(tasks_path.read_text(encoding="utf-8"))

task_results = experiment.get("task_results", [])
if not task_results:
    raise ValueError("No task_results found in experiment JSON.")

if len(task_results) > len(tasks):
    raise ValueError(
        f"task_results has {len(task_results)} items, but tasks file has {len(tasks)}."
    )

model_name = experiment_path.parent.name
print(f"Repo root: {repo_root}")
print(f"Loaded model: {model_name}")
print(f"Using {len(task_results)} task results.")

Repo root: c:\Users\Owner\OneDrive\Documents\Work\Robotics Research\LLM_Structure_Adherence
Loaded model: Qwen2.5-3B-Instruct
Using 10 task results.


In [8]:
import re

def ratio_str(numerator: int, denominator: int) -> str:
    if denominator == 0:
        return "0/0"
    return f"{numerator}/{denominator}"

def pct(numerator: int, denominator: int) -> float:
    if denominator == 0:
        return 0.0
    return (numerator / denominator) * 100.0

rows = []

# Overall row (from experiment.json)
overall_total_tasks = len(task_results)
overall_completed_tasks = sum(1 for tr in task_results if tr.get("task_completion", False))
overall_total_trees = sum(int(tr.get("bt_count", 0)) for tr in task_results)
overall_valid_trees = sum(int(tr.get("valid_structure_count", 0)) for tr in task_results)

rows.append(
    {
        "name": model_name,
        "task_accuracy": ratio_str(overall_completed_tasks, overall_total_tasks),
        "task_accuracy_pct": round(pct(overall_completed_tasks, overall_total_tasks), 2),
        "structure_adherence": ratio_str(overall_valid_trees, overall_total_trees),
        "structure_adherence_pct": round(pct(overall_valid_trees, overall_total_trees), 2),
    }
)

# Per-task-type rows from per-task files in the same model folder.
# This avoids relying on tasks_100 index order when a run is a custom subset.
model_dir = experiment_path.parent
task_files = sorted(model_dir.glob("task_*.json"))

by_type: dict[str, dict[str, int]] = {}
pattern = re.compile(r"^task_(.+)_v\d+\.json$")

for task_file in task_files:
    m = pattern.match(task_file.name)
    task_type = m.group(1) if m else "unknown"

    task_data = json.loads(task_file.read_text(encoding="utf-8"))
    bucket = by_type.setdefault(
        task_type,
        {
            "total_tasks": 0,
            "completed_tasks": 0,
            "total_trees": 0,
            "valid_trees": 0,
        },
    )

    bucket["total_tasks"] += 1
    bucket["completed_tasks"] += 1 if task_data.get("task_completion", False) else 0
    bucket["total_trees"] += int(task_data.get("bt_count", 0))
    bucket["valid_trees"] += int(task_data.get("valid_structure_count", 0))

for task_type in sorted(by_type.keys()):
    stats = by_type[task_type]
    rows.append(
        {
            "name": task_type,
            "task_accuracy": ratio_str(stats["completed_tasks"], stats["total_tasks"]),
            "task_accuracy_pct": round(pct(stats["completed_tasks"], stats["total_tasks"]), 2),
            "structure_adherence": ratio_str(stats["valid_trees"], stats["total_trees"]),
            "structure_adherence_pct": round(pct(stats["valid_trees"], stats["total_trees"]), 2),
        }
    )

df = pd.DataFrame(rows)
df

,name,task_accuracy,task_accuracy_pct,structure_adherence,structure_adherence_pct
0,Qwen2.5-3B-Instruct,2/10,20.0,35/44,79.55
1,face_target,1/2,50.0,6/6,100.00
2,go_around_obstacle,0/2,0.0,8/10,80.00
3,go_to_multiple_targets,0/2,0.0,8/10,80.00
4,go_to_target,1/2,50.0,7/8,87.50
5,move_to_closest_target,0/2,0.0,6/10,60.00
